**Step 3 of 3 — MongoDB Ingestion**

Transforms the enriched flat dataframe into nested MongoDB documents and bulk-inserts them into a MongoDB Atlas collection.

Each document follows the soft-schema structure:
```json
{
  "transaction_id": "txn_000001",
  "time": 3600,
  "amount": 149.62,
  "class": 0,
  "pca_features": { "V1": -1.359807, "V2": ..., "V28": ... },
  "cardholder": {
    "account_age_days": 730,
    "avg_monthly_spend": 842.50,
    "velocity_7d": 12
  },
  "merchant": {
    "mcc": "5411",
    "region": "Western Europe",
    "merchant_id": "merch_08821"
  }
}
```

**Run order:** ingest.ipynb → enrich.ipynb → load_mongo.ipynb

In [ ]:
#Install Dependencies
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 25.8 MB/s eta 0:00:00


In [ ]:
#Imports & Logging Setup
import logging
import os
import pandas as pd
from pymongo import MongoClient, errors, InsertOne
# Configure logging to both file and notebook output
logging.basicConfig(
    filename="load_mongo.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)
console = logging.StreamHandler()
console.setLevel(logging.INFO)
logging.getLogger("").addHandler(console)

In [ ]:
#connecting to DB
INPUT_PATH = "enriched.csv"
MONGO_URI = "connection_string_here"
#setting DB anme
DB_NAME = "fraud_detection"
#setting collection name
COLLECTION_NAME = "transactions"
#setting batch size
BATCH_SIZE = 1000
#PCA columns
PCA_COLS = [f"V{i}" for i in range(1, 29)]


In [ ]:
# Load enriched data
df = pd.read_csv("/content/enriched.csv")
logging.info(f"Loaded {len(df)} rows.")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V28,Amount,Class,transaction_id,cardholder_account_age_days,cardholder_avg_monthly_spend,cardholder_velocity_7d,merchant_mcc,merchant_region,merchant_id
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.021053,149.62,0,txn_000000,353,725.83,25,5651,Southern Europe,merch_09732
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0.014724,2.69,0,txn_000001,2832,8.24,38,5411,Eastern Europe,merch_01714
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.059752,378.66,0,txn_000002,2400,5398.55,31,5411,Northern Europe,merch_04005
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.061458,123.50,0,txn_000003,1619,672.25,16,5812,Eastern Europe,merch_06884
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0.215153,69.99,0,txn_000004,1597,95.80,6,4111,Southern Europe,merch_01580


In [ ]:
#Define Document Transformation
#Convert a single enriched row into the nested MongoDB document structure
def row_to_document(row: pd.Series) -> dict:
    return {
        "transaction_id": row["transaction_id"],
        "time": int(row["Time"]),
        "amount": round(float(row["Amount"]), 2),
        "class": int(row["Class"]),
        "pca_features": {
            col: round(float(row[col]), 6) for col in PCA_COLS
        },
        "cardholder": {
            "account_age_days": int(row["cardholder_account_age_days"]),
            "avg_monthly_spend": round(float(row["cardholder_avg_monthly_spend"]), 2),
            "velocity_7d": int(row["cardholder_velocity_7d"])
        },
        "merchant": {
            "mcc": str(row["merchant_mcc"]),
            "region": str(row["merchant_region"]),
            "merchant_id": str(row["merchant_id"])
        }
    }

In [ ]:
#Connect to MongoDB Atlas
logging.info(f"Connecting to MongoDB...")
try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    client.server_info()  # Trigger connection to catch auth errors early
    logging.info("Connected successfully.")
    print("Connected to MongoDB successfully.")
except errors.ServerSelectionTimeoutError as e:
    logging.error(f"Could not connect to MongoDB: {e}")
    raise

db = client[DB_NAME]
collection = db[COLLECTION_NAME]

Connected to MongoDB successfully.


In [ ]:
#Drop Existing Collection
collection.drop()
logging.info(f"Dropped existing '{COLLECTION_NAME}' collection.")
print(f"Dropped '{COLLECTION_NAME}' collection.")

Dropped 'transactions' collection.


In [ ]:
#Convert Rows to Documents
logging.info("Converting rows to documents...")
documents = [row_to_document(row) for _, row in df.iterrows()]
logging.info(f"{len(documents)} documents prepared.")
print(f"{len(documents)} documents ready for insertion.")

283726 documents ready for insertion.


In [ ]:
#Bulk Insert in Batches
def bulk_insert(collection, documents: list) -> int:
  #Insert a list of documents in bulk. Returns the number inserted.
    try:
        result = collection.bulk_write([InsertOne(doc) for doc in documents])
        return result.inserted_count
    except errors.BulkWriteError as e:
        logging.error(f"Bulk write error: {e.details}")
        raise

total_inserted = 0
num_batches = (len(documents) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(0, len(documents), BATCH_SIZE):
    batch = documents[i: i + BATCH_SIZE]
    inserted = bulk_insert(collection, batch)
    total_inserted += inserted
    batch_num = i // BATCH_SIZE + 1
    logging.info(f"Inserted batch {batch_num}/{num_batches}: {inserted} documents.")
    if batch_num % 50 == 0 or batch_num == num_batches:
        print(f"  Batch {batch_num}/{num_batches} — {total_inserted} total inserted")

logging.info(f"Load complete. Total documents inserted: {total_inserted}")

  Batch 50/284 — 50000 total inserted
  Batch 100/284 — 100000 total inserted
  Batch 150/284 — 150000 total inserted
  Batch 200/284 — 200000 total inserted
  Batch 250/284 — 250000 total inserted
  Batch 284/284 — 283726 total inserted


In [ ]:
#Close Connection
client.close()
logging.info("MongoDB connection closed.")
print("Connection closed. Pipeline complete.")

Connection closed. Pipeline complete.
